In [1]:
import torch
import torch.nn as nn
import timm
import pandas as pd
import numpy as np
import os
import math
import json
from PIL import Image
from collections import Counter
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm

device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

<jemalloc>: Unsupported system page size


Device: cuda:3


In [2]:
IMG_SIZE        = 224
BATCH_SIZE      = 32
ACCUM_STEPS     = 2
SAMPLE_FRAC     = 1.0          # ✅ ALLE Daten nutzen!
BACKBONE        = "eva02_base_patch14_224"
NUM_WORKERS     = 4
SEED            = 42

# EVA02 eigene Normalisierung
MEAN = [0.48145466, 0.4578275,  0.40821073]
STD  = [0.26862954, 0.26130258, 0.27577711]

torch.manual_seed(SEED)
np.random.seed(SEED)

In [3]:
data_dir   = "/datasets/multi-view-pig-posture-recognition/"
train2_df  = pd.read_csv(os.path.join(data_dir, "train2.csv"))
dir_train2 = os.path.join(data_dir, "train2_images")
test_df    = pd.read_csv(os.path.join(data_dir, "test.csv"))
dir_test   = os.path.join(data_dir, "test_images")

In [4]:
def parse_bbox_safe(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except:
            return [float(i) for i in x.replace("[","").replace("]","").split(",")]
    return x

train2_df["bbox"] = train2_df["bbox"].apply(parse_bbox_safe)
test_df["bbox"]   = test_df["bbox"].apply(parse_bbox_safe)

print(f"Train: {len(train2_df)}, Test: {len(test_df)}")
print(f"Klassenverteilung:\n{train2_df['class_id'].value_counts().sort_index()}")

def crop_pig(image, bbox, expand=1.2):
    """Schwein aus Bild ausschneiden mit Expansion"""
    x, y, w, h = bbox
    cx, cy     = x + w / 2, y + h / 2
    w_exp      = w * expand
    h_exp      = h * expand
    img_w, img_h = image.size
    x_min = max(0, int(cx - w_exp / 2))
    y_min = max(0, int(cy - h_exp / 2))
    x_max = min(img_w, int(cx + w_exp / 2))
    y_max = min(img_h, int(cy + h_exp / 2))
    if x_max > x_min and y_max > y_min:
        image = image.crop((x_min, y_min, x_max, y_max))
    return image

Train: 23450, Test: 11708
Klassenverteilung:
class_id
0    3083
1    3435
2     695
3    9928
4    6309
Name: count, dtype: int64


In [5]:
class PigDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform
        self.is_test   = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(os.path.join(self.img_dir, row["image_id"])).convert("RGB")
        image = crop_pig(image, row["bbox"], expand=1.2)

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image, row["row_id"]
        return image, int(row["class_id"])

In [6]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.2)),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, val_idx = next(skf.split(train2_df, train2_df["class_id"]))
train_df = train2_df.iloc[train_idx]
val_df   = train2_df.iloc[val_idx]
print(f"Train: {len(train_df)}, Val: {len(val_df)}")

Train: 18760, Val: 4690


In [7]:
train_labels  = train_df["class_id"].tolist()
class_counts  = Counter(train_labels)
num_classes   = len(class_counts)
total_samples = len(train_labels)

# Gewichte pro Sample (seltene Klassen öfter samplen)
sample_weights = [1.0 / class_counts[l] for l in train_labels]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# Loss-Gewichte (sqrt für sanftere Gewichtung)
class_weights_list   = [math.sqrt(total_samples / (num_classes * class_counts[i])) for i in range(num_classes)]
class_weights_tensor = torch.FloatTensor(class_weights_list).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.1)

train_dataset = PigDataset(train_df, dir_train2, transform=train_transform)
val_dataset   = PigDataset(val_df,   dir_train2, transform=val_transform)

# ✅ WeightedRandomSampler statt shuffle=True
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train Batches: {len(train_loader)}, Val Batches: {len(val_loader)}")


Train Batches: 586, Val Batches: 147


In [8]:
class PigModel(nn.Module):
    def __init__(self, num_classes, backbone_name=BACKBONE):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0)
        self.backbone.requires_grad_(False)
        feat_dim = self.backbone.num_features
        print(f"Feature dim: {feat_dim}")

        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Dropout(0.4),
            nn.Linear(feat_dim, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.head(self.backbone(x))

    def unfreeze(self, fraction=0.0):
        params = list(self.backbone.parameters())
        # Alles einfrieren, dann letzten fraction% auftauen
        for p in params:
            p.requires_grad = False
        if fraction > 0:
            n = int(len(params) * fraction)
            for p in params[-n:]:
                p.requires_grad = True
            print(f"Unfroze last {n}/{len(params)} backbone params ({fraction*100:.0f}%)")


In [9]:
model = PigModel(num_classes=num_classes)
model.to(device)
print(f"VRAM: {torch.cuda.memory_allocated(device)/1e9:.2f} GB")


Feature dim: 768
VRAM: 0.35 GB


In [10]:
class CosineWarmupScheduler:
    def __init__(self, optimizer, warmup_epochs, max_epochs, min_lr_ratio=0.01):
        self.optimizer     = optimizer
        self.warmup_epochs = warmup_epochs
        self.max_epochs    = max_epochs
        self.min_lr_ratio  = min_lr_ratio
        self.current_epoch = 0
        for pg in optimizer.param_groups:
            pg['initial_lr'] = pg['lr']

    def step(self):
        self.current_epoch += 1
        e = self.current_epoch
        if e <= self.warmup_epochs:
            scale = e / self.warmup_epochs
        else:
            progress = (e - self.warmup_epochs) / (self.max_epochs - self.warmup_epochs)
            scale = self.min_lr_ratio + 0.5 * (1 - self.min_lr_ratio) * (1 + math.cos(math.pi * progress))
        for pg in self.optimizer.param_groups:
            pg['lr'] = pg['initial_lr'] * scale

    def get_lr(self):
        return self.optimizer.param_groups[0]['lr']

def mixup(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_loss(criterion, pred, ya, yb, lam):
    return lam * criterion(pred, ya) + (1 - lam) * criterion(pred, yb)

# ✅ Kompatibel mit älteren und neueren PyTorch-Versionen
scaler = torch.cuda.amp.GradScaler()

def autocast():
    return torch.cuda.amp.autocast()

print(f"PyTorch version: {torch.__version__}")
print("✅ Scaler & autocast bereit")

PyTorch version: 1.12.1
✅ Scaler & autocast bereit


In [11]:
def train_epoch(model, loader, optimizer, use_mixup=True):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    for i, (imgs, lbls) in enumerate(tqdm(loader, desc="  Train", leave=False)):
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)

        if use_mixup and np.random.rand() < 0.5:
            imgs, ya, yb, lam = mixup(imgs, lbls, alpha=0.4)
            with autocast():
                loss = mixup_loss(criterion, model(imgs), ya, yb, lam) / ACCUM_STEPS
        else:
            with autocast():
                loss = criterion(model(imgs), lbls) / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (i + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * ACCUM_STEPS

    return total_loss / len(loader)

In [12]:
def val_epoch(model, loader):
    from sklearn.metrics import f1_score
    model.eval()
    correct, total, val_loss = 0, 0, 0.0
    all_preds, all_labels = [], []

    for imgs, lbls in tqdm(loader, desc="  Val  ", leave=False):
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)
        with autocast():
            out  = model(imgs)
            val_loss += criterion(out, lbls).item()
        preds = out.argmax(1)
        correct += (preds == lbls).sum().item()
        total   += lbls.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(lbls.cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return val_loss / len(loader), correct / total, macro_f1

def run_phase(model, train_loader, val_loader, optimizer, scheduler,
              num_epochs, save_name, use_mixup=True):
    best_acc = 0.0
    for epoch in range(num_epochs):
        tr_loss = train_epoch(model, train_loader, optimizer, use_mixup)
        va_loss, va_acc, va_f1 = val_epoch(model, val_loader)
        lr = scheduler.get_lr()
        scheduler.step()

        saved = ""
        if va_f1 > best_acc:
            best_acc = va_f1
            torch.save(model.state_dict(), save_name)
            saved = " ✅"

        print(f"  Ep {epoch+1:02d}/{num_epochs} | "
              f"TrL: {tr_loss:.4f} | VaL: {va_loss:.4f} | "
              f"Acc: {va_acc:.4f} | F1: {va_f1:.4f} | LR: {lr:.2e}{saved}")
    return best_acc

In [13]:
print("\n" + "="*50)
print("PHASE 1: Head only")
print("="*50)

opt1 = torch.optim.AdamW(model.head.parameters(), lr=1e-3, weight_decay=1e-2)
sch1 = CosineWarmupScheduler(opt1, warmup_epochs=1, max_epochs=5)

best = run_phase(model, train_loader, val_loader, opt1, sch1,
                 num_epochs=1, save_name="pig_phase1.pth", use_mixup=False)
print(f"  → Best: {best:.4f}")


PHASE 1: Head only


  Ep 01/1 | TrL: 1.2950 | VaL: 1.4972 | Acc: 0.3940 | F1: 0.3598 | LR: 1.00e-03 ✅
  → Best: 0.3598


In [15]:
print("\n" + "="*50)
print("PHASE 2: 25% Backbone")
print("="*50)

model.load_state_dict(torch.load("pig_phase1.pth", map_location=device))
model.unfreeze(0.25)

opt2 = torch.optim.AdamW([
    {"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": 5e-6},
    {"params": model.head.parameters(), "lr": 5e-5},
], weight_decay=5e-3)
sch2 = CosineWarmupScheduler(opt2, warmup_epochs=1, max_epochs=6)

best = run_phase(model, train_loader, val_loader, opt2, sch2,
                 num_epochs=1, save_name="pig_phase2.pth", use_mixup=True)
print(f"  → Best: {best:.4f}")



PHASE 2: 25% Backbone
Unfroze last 58/234 backbone params (25%)


  Ep 01/1 | TrL: 1.1816 | VaL: 1.2616 | Acc: 0.5610 | F1: 0.4853 | LR: 5.00e-06 ✅
  → Best: 0.4853


In [16]:
print("\n" + "="*50)
print("PHASE 3: 50% Backbone")
print("="*50)

model.load_state_dict(torch.load("pig_phase2.pth", map_location=device))
model.unfreeze(0.5)

opt3 = torch.optim.AdamW([
    {"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": 1e-6},
    {"params": model.head.parameters(), "lr": 1e-5},
], weight_decay=5e-3)
sch3 = CosineWarmupScheduler(opt3, warmup_epochs=1, max_epochs=6)

best = run_phase(model, train_loader, val_loader, opt3, sch3,
                 num_epochs=1, save_name="pig_phase3.pth", use_mixup=True)
print(f"  → Best: {best:.4f}")


PHASE 3: 50% Backbone
Unfroze last 117/234 backbone params (50%)


  Ep 01/1 | TrL: 1.0731 | VaL: 1.2634 | Acc: 0.5687 | F1: 0.5072 | LR: 1.00e-06 ✅
  → Best: 0.5072


In [17]:
print("\n" + "="*50)
print("PHASE 4: 80% Backbone (Fine-Tuning)")
print("="*50)

model.load_state_dict(torch.load("pig_phase3.pth", map_location=device))
model.unfreeze(0.8)

opt4 = torch.optim.AdamW([
    {"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": 2e-7},
    {"params": model.head.parameters(), "lr": 2e-6},
], weight_decay=1e-3)
sch4 = CosineWarmupScheduler(opt4, warmup_epochs=1, max_epochs=4)

best = run_phase(model, train_loader, val_loader, opt4, sch4,
                 num_epochs=1, save_name="pig_phase4.pth", use_mixup=True)
print(f"  → Best: {best:.4f}")



PHASE 4: 80% Backbone (Fine-Tuning)
Unfroze last 187/234 backbone params (80%)


  Ep 01/1 | TrL: 1.0558 | VaL: 1.2413 | Acc: 0.5642 | F1: 0.5026 | LR: 2.00e-07 ✅
  → Best: 0.5026


In [18]:
tta_transforms = [
    # Original
    transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    # Horizontal Flip
    transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.functional.hflip if False else transforms.Lambda(lambda x: x.transpose(Image.FLIP_LEFT_RIGHT)),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    # Vertical Flip
    transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.Lambda(lambda x: x.transpose(Image.FLIP_TOP_BOTTOM)),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    # Center Crop
    transforms.Compose([
        transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    # 90° Rotation
    transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.Lambda(lambda x: x.transpose(Image.ROTATE_90)),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
]

In [19]:
class TestDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(os.path.join(self.img_dir, row["image_id"])).convert("RGB")
        image = crop_pig(image, row["bbox"], expand=1.2)
        return self.transform(image), str(row["row_id"])

all_logits = {}

for t_idx, tta_tf in enumerate(tta_transforms):
    print(f"  TTA {t_idx+1}/{len(tta_transforms)} ...")
    loader = DataLoader(
        TestDataset(test_df, dir_test, tta_tf),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
    )
    with torch.no_grad():
        for imgs, rids in tqdm(loader, leave=False):
            imgs = imgs.to(device, non_blocking=True)
            with autocast():
                logits = model(imgs).float().cpu().numpy()
            for j, rid in enumerate(rids):
                all_logits[rid] = all_logits.get(rid, 0) + logits[j]


  TTA 1/5 ...


  TTA 2/5 ...


  TTA 3/5 ...


  TTA 4/5 ...


  TTA 5/5 ...
